# Powering Down Carbon? Exploring the Global Electricity Transition (2000–2025)

**Data Exploration and Preparation (DEP) project**
**Domain:** Energy
**Dataset:** Our World in Data – Energy dataset (Ritchie, Rosado and Roser, 2023)

## 1. Problem domain and motivation

### 1.1 The problem domain
Electricity and heat generation is the single largest source of energy-related carbon dioxide (CO₂)
emissions in the world (IEA, 2024). At the same time, electricity is the energy carrier through which
transport (electric vehicles) and heating (heat pumps) are expected to decarbonise. How a country
*generates* its electricity therefore sits at the centre of climate policy, including the Paris Agreement
goal of limiting warming to well below 2 °C (UNFCCC, 2015) and UN Sustainable Development Goal 7,
*"affordable, reliable, sustainable and modern energy for all"* (United Nations, 2015).

Countries start from very different positions. Some are rich in hydropower, some built nuclear fleets in
the 1970s–80s, and many still depend on coal. Their wealth, population size and demand for electricity also
differ by orders of magnitude. This makes the electricity transition a good subject for data exploration:
the data are large, multi-dimensional, heavily correlated and full of real-world quality problems.

### 1.2 Motivation
* **Policy relevance.** Knowing which countries are decarbonising, and which characteristics they share,
  helps judge whether the transition is happening fast enough and where it is stalling.
* **Data-preparation challenge.** The dataset joins several sources (Energy Institute, Ember, U.S. EIA,
  UN population, Maddison Project GDP) with different time coverage, so it contains structural missing
  values, aggregate (non-country) rows and extreme values that must be handled before any analysis.
* **Dimensionality.** There are 130 indicators, many of them near-duplicates (share, per-capita value and
  absolute value of the same source). This makes it a natural candidate for Principal Component Analysis.

### 1.3 Problem statement
> *How have national electricity systems changed between 2000 and 2025, and how are wealth (GDP per
> capita), population and energy demand related to the carbon intensity and the fuel mix of a country's
> electricity?*

### 1.4 Research questions
The exploratory analysis is organised around the following supporting questions:

| # | Question |
|---|---|
| Q1 | How has the global electricity mix (coal, gas, oil, hydro, nuclear, wind, solar) changed from 2000 to 2025? |
| Q2 | How are electricity demand, population and carbon intensity distributed across countries, and which countries are unusual? |
| Q3 | Is national wealth (GDP per capita) associated with cleaner electricity (lower carbon intensity)? |
| Q4 | Which continents have decarbonised their electricity fastest? |
| Q5 | Do energy sources substitute each other — e.g. does a high share of coal go with a low share of gas or of low-carbon sources? |
| Q6 | Can the many correlated indicators be summarised by a few principal components, and what do those components represent? |

## 2. The dataset chosen

| Item | Detail |
|---|---|
| Name | Our World in Data – Energy dataset (`owid-energy-data.csv`) |
| Publisher | Our World in Data (Ritchie, Rosado and Roser, 2023) |
| Location | <https://github.com/owid/energy-data> |
| Licence | Creative Commons BY 4.0 (free to reuse with attribution) |
| Unit of observation | One **country (or region) in one year** |
| Raw size | ≈ 23,000 rows × 130 columns, years 1900–2025 |
| Underlying sources | Energy Institute *Statistical Review of World Energy* (2025); Ember electricity data (2025); U.S. EIA international data; UN population estimates; Maddison Project GDP |

**Why this dataset?** It is openly licensed, well documented, updated every year, and wide enough to answer
questions about wealth, demand and the fuel mix together. Its known weaknesses — many missing values in
early years, overlapping regional aggregates and extreme values for very small or very large countries —
are exactly the issues a data-preparation project should deal with.

**Key units used in this project**

| Indicator family | Unit |
|---|---|
| `*_share_elec` | % of electricity generation coming from a source |
| `electricity_generation`, `*_electricity` | terawatt-hours (TWh) |
| `per_capita_electricity`, `electricity_demand_per_capita` | kilowatt-hours (kWh) per person |
| `energy_per_capita` | kWh of primary energy per person |
| `carbon_intensity_elec` | grams of CO₂-equivalent per kWh of electricity (gCO₂e/kWh) |
| `gdp` | international dollars (2011 prices, Maddison Project) |

## 3. Environment set-up

The cell below imports every library used in the notebook and sets common options so that results are
reproducible (fixed random seed) and figures look consistent. A small helper, `save_fig`, writes each key
figure to the `figures/` folder so it can be reused on the poster.

In [1]:
# --- Core data handling -------------------------------------------------------
import os                      # file paths and folder creation
import urllib.request          # download the raw data if it is not on disk
import warnings

import numpy as np             # numerical arrays
import pandas as pd            # tabular data (DataFrame)

# --- Visualisation ------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# --- Statistics and machine learning (used in later parts) ------------------------
import scipy
import sklearn

warnings.filterwarnings("ignore", category=FutureWarning)   # keep the output readable

# --- Reproducibility ----------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Display options: show more columns and round floats ----------------------
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

# --- Plot style: one consistent look for every figure ----------------------------
sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12})

# --- Folders -------------------------------------------------------------------
DATA_DIR = "data"
FIG_DIR = "figures"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, name):
    # Save a matplotlib figure as a high-resolution PNG in figures/ (used for the poster).
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path


# Print library versions so the environment can be reproduced
for lib in (np, pd, plt.matplotlib, sns, scipy, sklearn):
    print(f"{lib.__name__:<12}{lib.__version__}")

numpy       2.5.4
pandas      3.1.1
matplotlib  3.11.2
seaborn     0.13.2
scipy       1.18.1
sklearn     1.9.1


## 4. Loading the raw data

The raw CSV is stored in `data/`. If it is missing (for example on a fresh clone), it is downloaded from the
official Our World in Data GitHub repository. The raw file is **never modified**; every cleaning step later
works on a copy, so the full pipeline can be rerun from scratch.

In [2]:
RAW_URL = "https://raw.githubusercontent.com/owid/energy-data/master/owid-energy-data.csv"
RAW_PATH = os.path.join(DATA_DIR, "owid-energy-data.csv")

# Download only if the file is not already available locally
if not os.path.exists(RAW_PATH):
    print("Raw file not found - downloading from Our World in Data ...")
    urllib.request.urlretrieve(RAW_URL, RAW_PATH)

# Read the CSV into a DataFrame
raw = pd.read_csv(RAW_PATH)

print(f"File size : {os.path.getsize(RAW_PATH) / 1e6:.1f} MB")
print(f"Rows      : {raw.shape[0]:,}")
print(f"Columns   : {raw.shape[1]:,}")
print(f"Years     : {raw['year'].min()} - {raw['year'].max()}")
print(f"Entities  : {raw['country'].nunique():,} countries and regions")

File size : 9.2 MB
Rows      : 23,377
Columns   : 130
Years     : 1900 - 2025
Entities  : 314 countries and regions


In [3]:
# First rows: identifiers (country, year, iso_code) followed by the energy indicators
raw.head()

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,biofuel_electricity,biofuel_share_elec,biofuel_share_energy,carbon_intensity_elec,coal_cons_change_pct,coal_cons_change_twh,coal_cons_per_capita,coal_consumption,coal_elec_per_capita,coal_electricity,coal_prod_change_pct,coal_prod_change_twh,coal_prod_per_capita,coal_production,coal_share_elec,coal_share_energy,electricity_demand,electricity_demand_per_capita,electricity_generation,electricity_share_energy,...,other_renewables_energy_per_capita,other_renewables_share_elec,other_renewables_share_elec_exc_biofuel,other_renewables_share_energy,per_capita_electricity,primary_energy_consumption,renewables_cons_change_pct,renewables_cons_change_twh,renewables_consumption,renewables_elec_per_capita,renewables_electricity,renewables_energy_per_capita,renewables_share_elec,renewables_share_energy,solar_cons_change_pct,solar_cons_change_twh,solar_consumption,solar_elec_per_capita,solar_electricity,solar_energy_per_capita,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
0,ASEAN (Ember),2000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.870,1.550,NaN,572.580,NaN,NaN,NaN,NaN,NaN,76.030,NaN,NaN,NaN,NaN,20.073,NaN,378.760,NaN,378.760,NaN,...,NaN,6.020,4.470,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.230,NaN,19.334,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN
1,ASEAN (Ember),2001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.460,1.595,NaN,570.270,NaN,NaN,NaN,NaN,NaN,86.260,NaN,NaN,NaN,NaN,21.294,NaN,405.090,NaN,405.090,NaN,...,NaN,5.643,4.048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.190,NaN,19.055,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN
2,ASEAN (Ember),2002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.620,1.528,NaN,572.680,NaN,NaN,NaN,NaN,NaN,93.430,NaN,NaN,NaN,NaN,21.568,NaN,433.190,NaN,433.190,NaN,...,NaN,5.363,3.834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.530,NaN,17.667,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN
3,ASEAN (Ember),2003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.440,1.623,NaN,577.020,NaN,NaN,NaN,NaN,NaN,102.010,NaN,NaN,NaN,NaN,22.258,NaN,458.300,NaN,458.300,NaN,...,NaN,5.058,3.434,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.390,NaN,16.668,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN
4,ASEAN (Ember),2004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.390,1.689,NaN,583.640,NaN,NaN,NaN,NaN,NaN,115.010,NaN,NaN,NaN,NaN,23.159,NaN,496.610,NaN,496.610,NaN,...,NaN,5.026,3.337,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77.950,NaN,15.696,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN,NaN,NaN,NaN,NaN,0.000,NaN,0.000,NaN


In [4]:
# Most recent rows for one well-known country, to see what a 'typical' record looks like
raw.loc[raw["country"] == "United Kingdom",
        ["country", "year", "iso_code", "population", "electricity_generation",
         "coal_share_elec", "gas_share_elec", "wind_share_elec", "carbon_intensity_elec"]].tail()

,country,year,iso_code,population,electricity_generation,coal_share_elec,gas_share_elec,wind_share_elec,carbon_intensity_elec
21868,United Kingdom,2021,GBR,"67,668,788.000",307.900,2.205,39.896,21.088,267.100
21869,United Kingdom,2022,GBR,"68,179,315.000",324.580,1.830,38.545,24.712,254.980
21870,United Kingdom,2023,GBR,"68,682,965.000",292.530,1.292,34.752,28.083,235.630
21871,United Kingdom,2024,GBR,"69,138,185.000",284.160,0.669,30.370,29.307,216.500
21872,United Kingdom,2025,GBR,"69,551,334.000",292.310,0.113,31.101,29.366,217.410


### 4.1 First impressions
* The raw table is large (≈ 23,000 rows × 130 columns) and already in a **long, tidy format**: one row per
  entity per year.
* The column names follow a clear pattern, `<source>_<measure>` (e.g. `coal_share_elec`,
  `solar_electricity`), so the 130 columns are really a handful of measures repeated for each energy source.
* Even the first rows show many `NaN` values: they belong to a regional aggregate (*ASEAN (Ember)*) that has
  no population or GDP, only electricity figures. Missing data will be the main cleaning challenge.
* The `country` column contains not only countries but also regions and income groups (e.g. *World*,
  *Africa*, *High-income countries*). These must be separated from true countries to avoid double counting.

The next part characterises the raw data in detail and selects the subset of rows and columns used in the
rest of the project.